# Planars — {{LANG_DISPLAY}} Report

Generates a shareable HTML report for **{{LANG_DISPLAY}}**, displays it inline, and saves it to Google Drive.

**To use:** From the menu: **Runtime → Run all** — Google will ask for permission once.
The report renders in the cell below and the Drive URL printed at the end is stable: bookmark it and share it. Future runs refresh the same file.

In [ ]:
#@title Setup
import subprocess, sys, importlib

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
     'git+https://github.com/jcgood/planars.git',
     'gspread', 'google-auth'])
importlib.invalidate_caches()
import logging
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)

from google.colab import auth
auth.authenticate_user()

import json, gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

# CONFIG_FILE_ID, LANG_ID, and FOLDER_ID are baked in at notebook generation time.
# To update them, re-run: python -m coding generate-notebooks --apply
CONFIG_FILE_ID = '{{CONFIG_FILE_ID}}'
LANG_ID        = '{{LANG_ID}}'
FOLDER_ID      = '{{FOLDER_ID}}'

config   = json.loads(drive_svc.files().get_media(fileId=CONFIG_FILE_ID).execute())
manifest = {LANG_ID: config[LANG_ID]}

print('Ready ✓')

In [ ]:
#@title Generate and Upload Report
from planars.reports import language_report_data
from planars.html_report import render_language_report
from googleapiclient.http import MediaInMemoryUpload

print('Collecting data...')
data = language_report_data(LANG_ID, source='sheets', gc=gc, manifest=manifest)
n_spans = len(data['spans']) if data['spans'] is not None else 0
print(f'  {n_spans} span rows, {len(data["completeness"])} class(es)')

print('Rendering HTML...')
html = render_language_report(data)

print('Uploading to Drive...')
filename   = f'report_{LANG_ID}.html'
html_bytes = html.encode('utf-8')
media = MediaInMemoryUpload(html_bytes, mimetype='text/html', resumable=False)

query   = f"name='{filename}' and '{FOLDER_ID}' in parents and trashed=false"
results = drive_svc.files().list(q=query, fields='files(id)').execute()
existing = results.get('files', [])

if existing:
    file_id = existing[0]['id']
    drive_svc.files().update(fileId=file_id, media_body=media).execute()
    print('Updated existing report.')
else:
    result = drive_svc.files().create(
        body={'name': filename, 'parents': [FOLDER_ID]},
        media_body=media,
        fields='id',
    ).execute()
    file_id = result['id']
    drive_svc.permissions().create(
        fileId=file_id,
        body={'type': 'anyone', 'role': 'reader'},
        fields='id',
    ).execute()
    print('Created report (anyone with the link can view).')

print(f'\nReport URL: https://drive.google.com/file/d/{file_id}/view')
print('Bookmark this link \u2014 it stays stable on future runs.')

In [ ]:
#@title Display Report
from IPython.display import HTML, display
display(HTML(html))